# FlyDoom exploratory analysis

Runnable first-milestone checks for a connectome-constrained neural controller. This notebook uses anatomical topology as a computational graph; it does not model a fly's complete biological dynamics or subjective experience.

Sections follow the research workflow: runtime, ingestion, inspection, extraction, controls, memory, environment, encoding, sparse policy/PPO, evaluation, statistics, visualization, reproducibility, and validation.

In [ ]:
# 1. Configure Runtime and Reproducibility
# 2. Import Dependencies
from __future__ import annotations

import hashlib
import json
import random
import subprocess
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import scipy
import torch

from flydoom.data.controls import degree_preserving_rewire, erdos_renyi_matched, shuffle_weights
from flydoom.data.loaders import generate_mock_connectome, load_connectome
from flydoom.data.subgraph import SubgraphSpec, extract_subgraph
from flydoom.env.doom_env import MockDoomEnv
from flydoom.env.observations import FlyInspiredVisualEncoder, SimpleCNNEncoder
from flydoom.models.connectome_network import ConnectomeRateNetwork
from flydoom.models.graph_policy import ConnectomePolicy
from flydoom.training.checkpoint import save_checkpoint
from flydoom.training.ppo import PPO, PPOConfig
from flydoom.training.trainer import evaluate, train

SEED = 42
NODE_COUNT = 1_000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "raw" / "notebook_mock"
OUTPUT_DIR = ROOT / "outputs" / "notebook_smoke"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
commit = (
    subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    or None
)
versions = {
    "torch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "gymnasium": gym.__version__,
    "scipy": scipy.__version__,
    "cuda": torch.cuda.is_available(),
    "git_commit": commit,
}
versions

In [ ]:
# 3. Generate the Mock Connectome
# 4. Load and Validate Canonical Connectome Tables
# 5. Inspect Connectome Graph Statistics
# 6. Extract a Sensory-to-Descending Subgraph
# 7. Construct Matched Control Graphs
# 8. Estimate Sparse Model Memory
generated = generate_mock_connectome(NODE_COUNT, seed=SEED)
generated.neurons.to_csv(DATA_DIR / "neurons.csv", index=False)
generated.edges.to_csv(DATA_DIR / "synapses.csv", index=False)
graph = load_connectome(DATA_DIR / "neurons.csv", DATA_DIR / "synapses.csv")
subgraph = extract_subgraph(
    graph,
    SubgraphSpec(
        source_class="visual",
        target_class="descending",
        min_synapses=1,
        max_neurons=NODE_COUNT,
        num_hops=8,
    ),
)
edge_index, edge_weight = subgraph.sparse_tensors()
fingerprint = hashlib.sha256(
    (DATA_DIR / "neurons.csv").read_bytes() + (DATA_DIR / "synapses.csv").read_bytes()
).hexdigest()
sample_edges = subgraph.edges.nlargest(min(20_000, subgraph.edge_count), "synapse_count")
nx_graph = nx.from_pandas_edgelist(
    sample_edges,
    "pre_neuron_id",
    "post_neuron_id",
    edge_attr="synapse_count",
    create_using=nx.DiGraph,
)
erdos = erdos_renyi_matched(subgraph, SEED)
rewired = degree_preserving_rewire(subgraph, SEED, swaps_per_edge=1)
weight_shuffled = shuffle_weights(subgraph, SEED)
estimated_mb = (subgraph.edge_count * 20 + subgraph.node_count * 4 + 128 * 84 * 84 * 3) / 1024**2
assert estimated_mb < 8_000, "Reduce max_neurons or rollout size"
summary = {
    "nodes": subgraph.node_count,
    "edges": subgraph.edge_count,
    "synapses": float(subgraph.edges.synapse_count.sum()),
    "weak_components_sample": nx.number_weakly_connected_components(nx_graph),
    "strong_components_sample": nx.number_strongly_connected_components(nx_graph),
    "sample_clustering": nx.average_clustering(nx_graph.to_undirected()),
    "fingerprint": fingerprint,
    "estimated_mb": estimated_mb,
}
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
subgraph.edges.pre_neuron_id.value_counts().hist(bins=40, ax=axes[0])
subgraph.edges.synapse_count.hist(bins=20, ax=axes[1])
axes[0].set_title("Out-degree")
axes[1].set_title("Synapse counts")
summary

In [ ]:
# 9. Configure VizDoom or Mock Environment
# 10. Encode Visual Observations
# 11. Map Features onto Sensory Nodes
# 12. Implement the Sparse Connectome Rate Network
# 13. Map Descending Nodes to Actions
env = MockDoomEnv(size=42, max_steps=32)  # Replace via make_environment for installed VizDoom.
observation, _ = env.reset(seed=SEED)
batch = torch.as_tensor(observation).unsqueeze(0)
cnn_features = SimpleCNNEncoder(64)(batch)
fly_features = FlyInspiredVisualEncoder()(batch)
sensory = torch.tensor(subgraph.neurons.index[subgraph.neurons.is_sensory].tolist())
descending = torch.tensor(subgraph.neurons.index[subgraph.neurons.is_descending].tolist())
rate_network = ConnectomeRateNetwork(
    subgraph.node_count, edge_index, edge_weight, trainable_weights=True, normalize_incoming=True
)
policy = ConnectomePolicy(
    rate_network, sensory, descending, action_count=5, encoder="cnn", propagation_steps=5
).to(DEVICE)
logits, value = policy(batch.to(DEVICE))
(logits.sum() + value.sum()).backward()
assert logits.shape == (1, 5)
assert policy.action_readout.in_features == len(descending)
assert policy.network.edge_weight.grad is not None
{
    "cnn": cnn_features.shape,
    "fly_inspired": fly_features.shape,
    "sensory_nodes": len(sensory),
    "descending_nodes": len(descending),
    "logits": logits.detach().cpu(),
}

In [ ]:
# 14. Implement PPO Training Components
# 15. Run Smoke Training
# 16. Train Connectome and Erdos-Renyi Controllers
def make_policy(candidate):
    indices, weights = candidate.sparse_tensors()
    sensory_nodes = torch.tensor(candidate.neurons.index[candidate.neurons.is_sensory].tolist())
    descending_nodes = torch.tensor(
        candidate.neurons.index[candidate.neurons.is_descending].tolist()
    )
    network = ConnectomeRateNetwork(candidate.node_count, indices, weights)
    return ConnectomePolicy(network, sensory_nodes, descending_nodes, propagation_steps=5).to(
        DEVICE
    )


training_tables = []
trained = {}
for model_name, candidate in {"connectome": subgraph, "erdos_renyi": erdos}.items():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    candidate_policy = make_policy(candidate)
    algorithm = PPO(candidate_policy, PPOConfig(epochs=1, batch_size=16))
    candidate_env = MockDoomEnv(size=42, max_steps=32)
    rows = train(
        candidate_env, algorithm, total_steps=64, rollout_steps=32, seed=SEED, device=DEVICE
    )
    table = pd.DataFrame(rows).assign(model=model_name)
    assert np.isfinite(table.select_dtypes("number")).all().all()
    training_tables.append(table)
    trained[model_name] = (candidate_policy, algorithm)
metrics = pd.concat(training_tables, ignore_index=True)
metrics

In [ ]:
# 17. Evaluate Saved Checkpoints
# 18. Compare Learning Performance
# 19. Run Statistical Comparisons
from flydoom.experiments.statistics import compare_samples

evaluations = []
for model_name, (candidate_policy, algorithm) in trained.items():
    checkpoint_path = OUTPUT_DIR / model_name / "checkpoint.pt"
    save_checkpoint(
        checkpoint_path,
        candidate_policy,
        algorithm.optimizer,
        {"seed": SEED, "dataset_fingerprint": fingerprint},
    )
    rewards = evaluate(
        MockDoomEnv(size=42, max_steps=32), candidate_policy, 3, SEED + 10_000, DEVICE
    )
    evaluations.append(
        {
            "model": model_name,
            "mean_reward": np.mean(rewards),
            "median_reward": np.median(rewards),
            "std_reward": np.std(rewards),
            "rewards": rewards,
        }
    )
evaluation_table = pd.DataFrame(evaluations)
auc = (
    metrics.groupby("model")
    .apply(
        lambda frame: np.trapezoid(frame.episodic_reward, frame.environment_steps),
        include_groups=False,
    )
    .rename("reward_auc")
)
# Inferential tests require independent seed-level results. Run >=5 seeds before interpreting them.
if all(len(row["rewards"]) >= 5 for row in evaluations):
    statistical_comparison = compare_samples(
        np.asarray(evaluations[0]["rewards"]), np.asarray(evaluations[1]["rewards"])
    )
else:
    statistical_comparison = {"status": "insufficient independent seeds; need at least 5"}
evaluation_table, auc, statistical_comparison

In [ ]:
# 20. Visualize Graph and Neural Activity
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_name, frame in metrics.groupby("model"):
    axes[0].plot(frame.environment_steps, frame.episodic_reward, marker="o", label=model_name)
plot_graph = nx_graph.subgraph(list(nx_graph.nodes)[:80])
node_regions = subgraph.neurons.set_index("neuron_id").brain_region.to_dict()
region_palette = {
    "visual": "#457b9d",
    "optic": "#2a9d8f",
    "central": "#e9c46a",
    "navigation": "#f4a261",
    "descending": "#e76f51",
}
nx.draw_networkx(
    plot_graph,
    ax=axes[1],
    node_size=12,
    arrows=False,
    with_labels=False,
    node_color=[region_palette.get(node_regions.get(node), "#777777") for node in plot_graph],
)
axes[0].set(xlabel="Environment steps", ylabel="Reward", title="Learning comparison")
axes[0].legend(frameon=False)
axes[1].set_title("Sampled connectome structure")
plt.tight_layout()
plt.show()

In [ ]:
# 21. Save Reproducibility Artifacts
# 22. Validate Core Components
resolved_config = {
    "seed": SEED,
    "node_count": NODE_COUNT,
    "device": str(DEVICE),
    "ppo": {"steps": 64, "rollout_steps": 32, "epochs": 1},
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(resolved_config, indent=2))
(OUTPUT_DIR / "subgraph_metadata.json").write_text(json.dumps(summary, indent=2))
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
evaluation_table.to_json(OUTPUT_DIR / "evaluation.json", orient="records", indent=2)

assert subgraph.id_to_index == {
    identifier: index for index, identifier in enumerate(subgraph.neurons.neuron_id)
}
assert edge_index.shape == (2, subgraph.edge_count)
assert erdos.edge_count == subgraph.edge_count
assert (
    rewired.edges.pre_neuron_id.value_counts()
    .sort_index()
    .equals(subgraph.edges.pre_neuron_id.value_counts().sort_index())
)
assert all((OUTPUT_DIR / name / "checkpoint.pt").exists() for name in trained)
print("Core assertions passed. Run `pytest` from the repository root for the complete suite.")